In [67]:
import pandas as pd

# Load the validated dataset
df = pd.read_csv("data/validated_full.csv", encoding="ISO-8859-1", low_memory=False)

print("Validated dataset loaded successfully!")
print("Shape of dataset:", df.shape)
df.head()


Validated dataset loaded successfully!
Shape of dataset: (536641, 8)


,InvoiceNo,StockCode,Description,Quantity,InvoiceDate,UnitPrice,CustomerID,Country
0,536365,85123A,WHITE HANGING HEART T-LIGHT HOLDER,6,12/1/2010 8:26,2.55,17850.0,United Kingdom
1,536365,71053,WHITE METAL LANTERN,6,12/1/2010 8:26,3.39,17850.0,United Kingdom
2,536365,84406B,CREAM CUPID HEARTS COAT HANGER,8,12/1/2010 8:26,2.75,17850.0,United Kingdom
3,536365,84029G,KNITTED UNION FLAG HOT WATER BOTTLE,6,12/1/2010 8:26,3.39,17850.0,United Kingdom
4,536365,84029E,RED WOOLLY HOTTIE WHITE HEART.,6,12/1/2010 8:26,3.39,17850.0,United Kingdom


In [68]:
before = df.shape[0]
df.drop_duplicates(inplace=True)
after = df.shape[0]
print(f"Removed {before - after} duplicate rows.")


Removed 0 duplicate rows.


In [69]:
df = df.copy()  # ensures df is not a view of another dataframe

# Fill missing values safely (no warnings)
df['CustomerID'] = df['CustomerID'].fillna('Unknown').astype(str)
df['Description'] = df['Description'].fillna('No description')


In [70]:
# Replace negative quantities with their absolute value
neg_count = (df['Quantity'] < 0).sum()
df.loc[df['Quantity'] < 0, 'Quantity'] = df['Quantity'].abs()
print(f"Corrected {neg_count} negative quantity values.")


Corrected 10587 negative quantity values.


In [71]:
# Convert InvoiceDate to datetime type
df['InvoiceDate'] = pd.to_datetime(df['InvoiceDate'], errors='coerce')

# Extract useful components
df['InvoiceYear'] = df['InvoiceDate'].dt.year
df['InvoiceMonth'] = df['InvoiceDate'].dt.month


In [72]:
# Calculate total cost per transaction
df['TotalCost'] = df['Quantity'] * df['UnitPrice']


In [73]:
# Make all column names lowercase and replace spaces with underscores
df.columns = [col.strip().lower().replace(' ', '_') for col in df.columns]


In [74]:
# Make a clean copy to avoid view warnings
inc = inc.copy()

# Normalize column names to lowercase (prevents KeyErrors due to capitalization)
inc.columns = [col.strip().lower().replace(" ", "_") for col in inc.columns]

# Check current columns
print("Columns in incremental dataset:", inc.columns.tolist())

# Fill missing values safely
if 'customerid' in inc.columns:
    inc['customerid'] = inc['customerid'].fillna('Unknown').astype(str)
if 'description' in inc.columns:
    inc['description'] = inc['description'].fillna('No description')

# Correct negative quantities
if 'quantity' in inc.columns:
    inc.loc[inc['quantity'] < 0, 'quantity'] = inc['quantity'].abs()

# Convert InvoiceDate to datetime
if 'invoicedate' in inc.columns:
    inc['invoicedate'] = pd.to_datetime(inc['invoicedate'], errors='coerce')

# Calculate total cost
if {'quantity', 'unitprice'}.issubset(inc.columns):
    inc['totalcost'] = inc['quantity'] * inc['unitprice']

# Standardize column names again for consistency
inc.columns = [col.strip().lower().replace(' ', '_') for col in inc.columns]

# Save the transformed incremental dataset
inc.to_csv("transformed/transformed_incremental.csv", index=False)
print("Transformed incremental dataset saved successfully!")


Columns in incremental dataset: ['invoiceno', 'stockcode', 'description', 'quantity', 'invoicedate', 'unitprice', 'customerid', 'country', 'totalcost']
Transformed incremental dataset saved successfully!


## Transform Phase Summary

This notebook performs the Transform phase of the ETL process.

### Transformations Applied
1. Removed duplicate rows.
2. Filled missing values in CustomerID and Description.
3. Converted negative Quantity values to positive.
4. Converted InvoiceDate to datetime format and extracted year and month.
5. Added a new column TotalCost = Quantity × UnitPrice.
6. Standardized column names (lowercase and underscores).

### Outputs
Transformed datasets saved to:
- transformed/transformed_full.csv
- transformed/transformed_incremental.csv
